# RQ1: Onset in 2025 from the KNMI 1 km temperature maps

This notebook computes the temperature-driven onset of the Betula pollen season in 2025 for every 1 km cell of the KNMI High-Resolution Maps (T_base = 3.5 °C from 1 March, shared threshold T* = 129.2 °C-days), compares it with the 33 KNMI stations and the city sensors in Enschede, and makes the map figure in ArcGIS Pro.

**Data (not included in this repository):** hourly KNMI High-Resolution Maps (.nc) for March and April 2025, the station onsets (`rq1_onset_stations.csv`), the Enschede sensor locations (`enschede_sensors.csv`) and a geodatabase with the station points, the Dutch border and the provinces. Steps 7 and 8 run in an ArcGIS Pro notebook.

## Step 1: Settings
Set the file paths and the model settings.

In [ ]:
import os, re, glob, csv
import datetime as dt
import numpy as np
import netCDF4

NC_DIRS = [os.path.join('data', 'hrm', '2025_03'), os.path.join('data', 'hrm', '2025_04')]   # hourly maps, subfolders included
OUT_DIR = 'output_rq1_hrm'
STATIONS = os.path.join('data', 'rq1_onset_stations.csv')
SENSORS = os.path.join('data', 'enschede_sensors.csv')
YEAR = 2025
T_BASE = 3.5        # °C
T_STAR = 129.2      # °C-days, shared threshold of Helmond and Leiden
MIN_HOURS = 18      # a day counts as complete with at least 18 of 24 hours
VAR = 'mean_DryBulbTemperature_Celsius'
MASKVAR = 'mean_DryBulbTemperature_Celsius_masked'   # NaN outside land
os.makedirs(OUT_DIR, exist_ok=True)

## Step 2: Daily maximum and minimum temperature
Read the hourly maps and keep the daily maximum and minimum per cell.

In [ ]:
files = []
for folder in NC_DIRS:
    files += glob.glob(os.path.join(folder, '**', '*.nc'), recursive=True)
files = sorted(set(files), key=lambda p: os.path.basename(p))
print(f'{len(files)} hourly files')
assert files, 'No .nc files found, check NC_DIRS.'

days = {}
lat = lon = landmask = None
for i, path in enumerate(files):
    t = re.search(r'(\d{12})\.nc$', os.path.basename(path))
    if not t:
        continue
    time = dt.datetime.strptime(t.group(1), '%Y%m%d%H%M')
    try:
        ds = netCDF4.Dataset(path)
    except Exception as e:
        print('Skipped (damaged file):', os.path.basename(path), e)
        continue
    a = np.array(ds[VAR][0], dtype='float32')
    if lat is None:
        lat = np.array(ds['y'][:]); lon = np.array(ds['x'][:])
        landmask = np.isfinite(np.array(ds[MASKVAR][0]))
    ds.close()
    d = time.date()
    if d not in days:
        days[d] = dict(tmax=a.copy(), tmin=a.copy(), n=1)
    else:
        np.fmax(days[d]['tmax'], a, out=days[d]['tmax'])
        np.fmin(days[d]['tmin'], a, out=days[d]['tmin'])
        days[d]['n'] += 1

## Step 3: Daily series and gap filling
Build the daily mean temperature from 1 March to 30 April and fill incomplete days by linear interpolation.

In [ ]:
start, end = dt.date(YEAR, 3, 1), dt.date(YEAR, 4, 30)
series = [start + dt.timedelta(days=k) for k in range((end - start).days + 1)]
hours = np.array([days[d]['n'] if d in days else 0 for d in series])
complete = hours >= MIN_HOURS
print(f'Days: {len(series)}, complete: {complete.sum()}, filled: {(~complete).sum()}')
print('Incomplete days:', [f'{d:%d %b} ({u} h)' for d, u, g in zip(series, hours, complete) if not g])

ny, nx = len(lat), len(lon)
tmean = np.full((len(series), ny, nx), np.nan, dtype='float32')
for k, d in enumerate(series):
    if complete[k]:
        tmean[k] = (days[d]['tmax'] + days[d]['tmin']) / 2.0
idx = np.where(complete)[0]
for k in np.where(~complete)[0]:
    left = idx[idx < k]; right = idx[idx > k]
    if len(left) and len(right):
        a, b = left[-1], right[0]; w = (k - a) / (b - a)
        tmean[k] = (1 - w) * tmean[a] + w * tmean[b]
    elif len(left):
        tmean[k] = tmean[left[-1]]
    elif len(right):
        tmean[k] = tmean[right[0]]

## Step 4: Onset per cell
Accumulate the GDD per cell and take the first day on which T* is reached (day 1 = 1 March).

In [ ]:
gdd = np.clip(tmean - T_BASE, 0, None)
agdd = np.nancumsum(gdd, axis=0)
reached = agdd >= T_STAR
onset = np.where(reached.any(axis=0), reached.argmax(axis=0) + 1, np.nan).astype('float32')
onset[~landmask] = np.nan
n_missing = int((~np.isfinite(onset) & landmask).sum())
if n_missing:
    print(f'{n_missing} land cells do not reach T* before 30 April.')
print(f'Onset (land): min {np.nanmin(onset):.0f}, max {np.nanmax(onset):.0f}, mean {np.nanmean(onset):.1f}')
np.savez_compressed(os.path.join(OUT_DIR, f'hrm_onset_{YEAR}.npz'), onset=onset, lat=lat, lon=lon,
                    landmask=landmask, hours=hours)

## Step 5: Comparison with the KNMI stations
Compare the onset relative to the national mean in the station cells with the onset from the 33 KNMI stations, and measure the spread within blocks of about 10 km.

In [ ]:
def cell(la, lo):
    # nearest land cell (coastal stations sometimes fall in a sea cell)
    i0 = np.abs(lat - la).argmin(); j0 = np.abs(lon - lo).argmin()
    for r in range(0, 6):
        sub = landmask[max(i0-r, 0):i0+r+1, max(j0-r, 0):j0+r+1]
        if sub.any():
            ii, jj = np.argwhere(sub)[0]
            return max(i0-r, 0) + ii, max(j0-r, 0) + jj
    return i0, j0

rows = []
with open(STATIONS, newline='', encoding='utf-8') as f:
    for r in csv.DictReader(f):
        k = r.get(f'on_{YEAR}', '')
        if k in ('', 'nan'):
            continue
        i, j = cell(float(r['lat']), float(r['lon']))
        rows.append(dict(name=r['naam'], lat=float(r['lat']), lon=float(r['lon']),
                         on_knmi=float(k), on_hrm=float(onset[i, j])))
on_k = np.array([r['on_knmi'] for r in rows]); on_h = np.array([r['on_hrm'] for r in rows])
ref_h = np.nanmean(on_h); ref_k = on_k.mean()
dev_k = on_k - ref_k; dev_h = on_h - ref_h
bias = np.nanmean(on_h - on_k)
rmse_dev = np.sqrt(np.nanmean((dev_h - dev_k) ** 2))
r_dev = np.corrcoef(dev_k[np.isfinite(dev_h)], dev_h[np.isfinite(dev_h)])[0, 1]
print(f'{len(rows)} stations: onset 1 km minus station {bias:+.1f} days')
print(f'Deviation from the national mean: r = {r_dev:.2f}, RMSE = {rmse_dev:.1f} days')
with open(os.path.join(OUT_DIR, f'stations_knmi_vs_hrm_{YEAR}.csv'), 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['name', 'lat', 'lon', 'onset_knmi', 'onset_hrm', 'dev_knmi', 'dev_hrm'])
    for r, a, b in zip(rows, dev_k, dev_h):
        w.writerow([r['name'], r['lat'], r['lon'], r['on_knmi'], r['on_hrm'], round(a, 2), round(b, 2)])

dev_map = onset - ref_h
print(f'1 km map: 5th to 95th percentile {np.nanpercentile(dev_map, 5):+.1f} to '
      f'{np.nanpercentile(dev_map, 95):+.1f} days, SD {np.nanstd(dev_map):.1f}')

# spread within blocks of 10 x 10 cells
block = 10
ranges, sd_block, p1090 = [], [], []
for i in range(0, ny - block, block):
    for j in range(0, nx - block, block):
        b = dev_map[i:i+block, j:j+block]
        b = b[np.isfinite(b)]
        if b.size > 50:
            ranges.append(b.max() - b.min())
            sd_block.append(b.std())
            p1090.append(np.percentile(b, 90) - np.percentile(b, 10))
print(f'Within about 10 km: range median {np.median(ranges):.0f} days, SD median {np.median(sd_block):.1f} days, '
      f'P10 to P90 median {np.median(p1090):.1f} days, 90th percentile {np.percentile(p1090, 90):.1f} days '
      f'(n = {len(ranges)} blocks)')

## Step 6: Enschede
Compare the onset in the 1 km cells of the city sensors with the cell of KNMI Twenthe.

In [ ]:
with open(SENSORS, newline='', encoding='utf-8') as f:
    sens = [s for s in csv.DictReader(f) if s.get('type', 'sensor') == 'sensor']
values = []
for s in sens:
    i, j = cell(float(s['lat']), float(s['lon']))
    values.append(onset[i, j])
    print(f"{s['naam']:22s} cell ({i},{j})  onset {onset[i, j]:.0f}   sensor shift {YEAR}: {s.get(f'shift_{YEAR}', '')}")
it, jt = cell(52.27, 6.89)
print(f"{'KNMI Twenthe':22s} cell ({it},{jt})  onset {onset[it, jt]:.0f}")
print(f'City cells mean {np.nanmean(values):.1f}, Twenthe {onset[it, jt]:.0f}, '
      f'difference {onset[it, jt] - np.nanmean(values):+.1f} days')

## Step 7: Rasters for ArcGIS
Save the onset and the deviation from the national mean as GeoTIFF (requires arcpy).

In [ ]:
import arcpy
ll = arcpy.Point(float(lon.min()) - 0.005, float(lat.min()) - 0.005)
for nm, arr in (('hrm_onset', onset), ('hrm_dev', dev_map)):
    a = arr[::-1] if lat[0] < lat[-1] else arr   # row 0 must be the northern edge
    ras = arcpy.NumPyArrayToRaster(np.nan_to_num(a, nan=-9999).astype('float32'), ll, 0.01, 0.01, -9999)
    out = os.path.join(OUT_DIR, f'{nm}_{YEAR}.tif')
    ras.save(out)
    arcpy.management.DefineProjection(out, arcpy.SpatialReference(4326))
    print('Raster:', out)

## Step 8: Map figure
Make the layout with (a) the interpolated station onsets (empirical Bayesian kriging), (b) the 1 km map and (c) Enschede with the city sensors and KNMI Twenthe.

In [ ]:
from arcpy.sa import ExtractByMask
arcpy.env.overwriteOutput = True
arcpy.env.addOutputsToMap = False
arcpy.CheckOutExtension('GeoStats'); arcpy.CheckOutExtension('Spatial')
aprx = arcpy.mp.ArcGISProject('CURRENT')
WORK = aprx.defaultGeodatabase

GDB = os.path.abspath(os.path.join('data', 'thesis.gdb'))
POINTS = os.path.join(GDB, 'onset_rd')              # KNMI stations with onset deviation per year
BORDER = os.path.join(GDB, 'nederland_dissolve')    # outline of the Netherlands
PROV = os.path.join(GDB, 'provincies_rd')           # provinces
HRM = os.path.abspath(os.path.join(OUT_DIR, f'hrm_dev_{YEAR}.tif'))
SENS = os.path.abspath(SENSORS)
FIELD = f'afw_{YEAR}'
OUT_PNG = os.path.abspath(os.path.join(OUT_DIR, f'RQ1_HRM_{YEAR}.png'))
for p in (POINTS, BORDER, PROV, HRM, SENS):
    assert arcpy.Exists(p) or os.path.exists(p), f'Not found: {p}'
SR = arcpy.Describe(POINTS).spatialReference
arcpy.env.outputCoordinateSystem = SR

# classes and colours, the same as in the station map
BREAKS = [-6, -4, -2, 0, 2, 4, 6, 40]
LABELS = ['6 or more days earlier', '4 to 6 earlier', '2 to 4 earlier', '0 to 2 earlier',
          '0 to 2 later', '2 to 4 later', '4 to 6 later', '6 or more days later']
COLOURS = [[178, 24, 43], [214, 96, 77], [244, 165, 130], [253, 219, 199],
           [209, 229, 240], [146, 197, 222], [67, 147, 195], [33, 102, 172]]
RASTER_NAME = f'Onset relative to national mean, {YEAR}'

def to_float(x):
    return float(str(x).replace(',', '.'))

# 1. EBK of the KNMI stations
arcpy.management.MakeFeatureLayer(POINTS, 'sel', f'{FIELD} IS NOT NULL')
tmp = os.path.join(WORK, 'tmp_knmi')
res = arcpy.ga.EmpiricalBayesianKriging(in_features='sel', z_field=FIELD, out_ga_layer='ga_knmi',
                                        out_raster=tmp, cell_size=1000)
knmi_r = os.path.join(WORK, f'knmi_dev_{YEAR}')
arcpy.env.extent = arcpy.Describe(BORDER).extent
ExtractByMask(tmp, BORDER).save(knmi_r)
arcpy.management.Delete(tmp)
try:
    cv = arcpy.ga.CrossValidation(res.getOutput(0))
    print(f"EBK: n = {arcpy.management.GetCount('sel')[0]}, RMSE = {to_float(cv.rootMeanSquare):.2f} days")
except Exception as e:
    print('Cross validation failed:', e)

# 2. 1 km map projected to RD New and clipped to the Netherlands
arcpy.env.extent = 'MAXOF'
hrm_tmp = os.path.join(WORK, 'hrm_dev_rd_tmp')
arcpy.management.ProjectRaster(HRM, hrm_tmp, SR, 'NEAREST', '1000')
hrm_r = os.path.join(WORK, f'hrm_dev_{YEAR}_rd')
ExtractByMask(hrm_tmp, BORDER).save(hrm_r)
arcpy.management.Delete(hrm_tmp)

# 3. City sensors as points
sens_wgs = os.path.join(WORK, 'sens_wgs')
arcpy.management.XYTableToPoint(SENS, sens_wgs, 'lon', 'lat', coordinate_system=arcpy.SpatialReference(4326))
sens_rd = os.path.join(WORK, 'enschede_sensors_rd')
arcpy.management.Project(sens_wgs, sens_rd, SR)
arcpy.management.Delete(sens_wgs)
fields = [f.name for f in arcpy.ListFields(sens_rd)]
sens_query = "type = 'sensor'" if 'type' in fields else None

# 4. Maps
def set_colours(rl):
    sym = rl.symbology
    sym.updateColorizer('RasterClassifyColorizer')
    c = sym.colorizer
    c.classificationField = 'Value'; c.classificationMethod = 'ManualInterval'
    c.breakCount = len(BREAKS)
    for i, brk in enumerate(c.classBreaks):
        brk.upperBound = BREAKS[i]; brk.label = LABELS[i]; brk.color = {'RGB': COLOURS[i] + [100]}
    rl.symbology = sym

def style_points(lyr, colour, size):
    try:
        s = lyr.symbology
        s.renderer.symbol.color = {'RGB': colour + [100]}
        s.renderer.symbol.size = size
        lyr.symbology = s
    except Exception as e:
        print('Point symbol not changed:', e)

def style_provinces(lyr):
    try:
        s = lyr.symbology
        s.renderer.symbol.color = {'RGB': [0, 0, 0, 0]}
        s.renderer.symbol.outlineColor = {'RGB': [120, 120, 120, 100]}
        s.renderer.symbol.outlineWidth = 0.4
        lyr.symbology = s
    except Exception as e:
        print('Province symbol not changed:', e)

def new_map(name):
    for m in aprx.listMaps(name):
        aprx.deleteItem(m)
    m = aprx.createMap(name, 'MAP')
    m.spatialReference = SR
    for lyr in m.listLayers():
        if lyr.isBasemapLayer:
            m.removeLayer(lyr)
    return m

LAYOUT = f'RQ1_HRM_{YEAR}'
for l in aprx.listLayouts(LAYOUT):
    aprx.deleteItem(l)

maps = {}
for key, raster in (('a', knmi_r), ('b', hrm_r), ('c', hrm_r)):
    m = new_map(f'RQ1_HRM_{key}')
    rl = m.addDataFromPath(raster); rl.name = RASTER_NAME; set_colours(rl)
    pv = m.addDataFromPath(PROV); pv.name = 'Provinces'; style_provinces(pv)
    if key in ('a', 'b'):
        st = m.addDataFromPath(POINTS); st.name = 'KNMI weather station'
        st.definitionQuery = f'{FIELD} IS NOT NULL'; style_points(st, [40, 40, 40], 3)
    else:
        se = m.addDataFromPath(sens_rd); se.name = 'City temperature sensor'
        if sens_query:
            se.definitionQuery = sens_query
        style_points(se, [200, 0, 160], 7)
        tw = m.addDataFromPath(POINTS); tw.name = 'KNMI Twenthe'
        tw.definitionQuery = "naam LIKE 'Twenthe%'"; style_points(tw, [255, 160, 0], 10)
    maps[key] = m

# 5. Layout (A4 portrait)
W, H, margin, gap = 21.0, 29.7, 1.0, 0.4
lay = aprx.createLayout(W, H, 'CENTIMETER', LAYOUT)
w = (W - 2 * margin - gap) / 2
h_top = 12.5
y_top = H - margin - h_top
h_bot = y_top - gap - margin - 0.2
frames = {}
pos = {'a': (margin, y_top, w, h_top), 'b': (margin + w + gap, y_top, w, h_top),
       'c': (margin, margin, w, h_bot)}
titles = {'a': 'a) 33 KNMI stations, interpolated (EBK)',
          'b': 'b) KNMI high-resolution temperature maps (1 km)',
          'c': 'c) Enschede: 1 km cells of (b), city sensors\n    and KNMI Twenthe'}
for k, (x0, y0, ww, hh) in pos.items():
    mf = lay.createMapFrame(arcpy.Extent(x0, y0, x0 + ww, y0 + hh), maps[k], f'Frame_{k}')
    frames[k] = mf
    t = aprx.createTextElement(lay, arcpy.Point(x0 + 0.2, y0 + hh - (0.85 if '\n' in titles[k] else 0.5)),
                               'POINT', titles[k], 7)
    t.name = f'Title_{k}'

ext_nl = arcpy.Describe(BORDER).extent
for k in ('a', 'b'):
    frames[k].camera.setExtent(ext_nl)
scale = max(frames['a'].camera.scale, frames['b'].camera.scale) * 1.03
frames['a'].camera.scale = scale; frames['b'].camera.scale = scale

# Enschede: sensors and Twenthe with a 1.5 km margin
xs, ys = [], []
with arcpy.da.SearchCursor(sens_rd, ['SHAPE@XY'], sens_query) as cur:
    for (x, y), in cur:
        xs.append(x); ys.append(y)
with arcpy.da.SearchCursor(POINTS, ['SHAPE@XY'], "naam LIKE 'Twenthe%'") as cur:
    for (x, y), in cur:
        xs.append(x); ys.append(y)
r = 1500
frames['c'].camera.setExtent(arcpy.Extent(min(xs) - r, min(ys) - r, max(xs) + r, max(ys) + r))

# legend, north arrow and scale bars
if 'ArcGIS 2D' not in aprx.styles:
    aprx.updateStyles(list(aprx.styles) + ['ArcGIS 2D'])

def style_item(kind, preferred):
    items = aprx.listStyleItems('ArcGIS 2D', kind, preferred) or aprx.listStyleItems('ArcGIS 2D', kind)
    return items[0] if items else None

def surround(geom, kind, preferred, frame, name):
    st = style_item(kind, preferred)
    if st is not None:
        try:
            return lay.createMapSurroundElement(geom, kind, frame, st, name)
        except Exception:
            pass
    return lay.createMapSurroundElement(geom, kind, frame, name=name)

xr = margin + w + gap
surround(arcpy.Extent(xr, margin + 3.2, xr + w, y_top - gap - 0.2), 'LEGEND', 'Legend 1', frames['b'], 'Leg_b')
surround(arcpy.Extent(xr, margin, xr + w - 2.5, margin + 3.0), 'LEGEND', 'Legend 1', frames['c'], 'Leg_c')
surround(arcpy.Point(W - margin - 1.0, margin + 1.2), 'NORTH_ARROW', 'ArcGIS North 1', frames['b'], 'North')
surround(arcpy.Point(xr, y_top - 0.2), 'SCALE_BAR', 'Scale Line 1 Metric', frames['b'], 'Scale_b')
surround(arcpy.Point(margin + 0.3, margin + 0.3), 'SCALE_BAR', 'Scale Line 1 Metric', frames['c'], 'Scale_c')

# legends: no provinces, no 'Value' heading, colour classes only in the main legend
cim = lay.getDefinition('V3')
for el in cim.elements:
    if type(el).__name__ != 'CIMLegend':
        continue
    el.title = 'Legend' if el.name == 'Leg_b' else ''
    keep = []
    for it in el.items:
        if it.name == 'Provinces':
            continue
        if el.name == 'Leg_c' and it.name == RASTER_NAME:
            continue
        it.showHeading = False
        keep.append(it)
    el.items = keep
lay.setDefinition(cim)

aprx.save()
lay.exportToPNG(OUT_PNG, resolution=300)
print('Layout:', LAYOUT, '| PNG:', OUT_PNG)